# PRÁCTICA CLASE 3
 BERT, RoBERTa y DistilBERT para análisis de sentimientos
 en español usando HuggingFace Transformers.

 Incluye:
 1. Dataset de sentimientos en español.
 2. Fine-tuning de BETO, RoBERTa y DistilBETO.
 3. Evaluación con accuracy y F1 macro.
 4. Comparación de rendimiento.
 5. Curvas de aprendizaje.
 6. Predicción con textos nuevos.

 Instalación recomendada:
 pip install -U torch transformers datasets accelerate scikit-learn pandas matplotlib

In [1]:
# Importar dependencias
import time
import random
import inspect
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from datasets import Dataset, DatasetDict, load_dataset
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    pipeline,
    set_seed,
)

c:\Users\jpach\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ============================================================
# 1. CONFIGURACIÓN GENERAL
# ============================================================

SEED = 666
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo detectado:", DEVICE)

# Útil para equipos con CPU o Windows.
torch.set_num_threads(1)

OUTPUT_BASE_DIR = Path("resultados_practica_bert")
OUTPUT_BASE_DIR.mkdir(exist_ok=True)

# Modo rápido para clase.
# Cambiar a False para una ejecución más completa.
FAST_MODE = True

# Para una práctica en clase se recomienda False, porque evita
# problemas cuando un dataset externo cambia columnas o etiquetas.
# Si quieres intentar cargar TASS/IberBench desde HuggingFace,
# cambia esta variable a True.
USAR_DATASET_EXTERNO = False

MAX_LENGTH = 96

if FAST_MODE:
    NUM_EPOCHS = 1
    MAX_TRAIN_SAMPLES = 120
    MAX_EVAL_SAMPLES = 60
    TRAIN_BATCH_SIZE = 8
    EVAL_BATCH_SIZE = 16
else:
    NUM_EPOCHS = 3
    MAX_TRAIN_SAMPLES = 1000
    MAX_EVAL_SAMPLES = 300
    TRAIN_BATCH_SIZE = 8
    EVAL_BATCH_SIZE = 16

In [ ]:
# ============================================================
# 2. MODELOS A COMPARAR
# ============================================================

MODELS = {
    "BETO_BERT": "dccuchile/bert-base-spanish-wwm-cased",
    "RoBERTa_ES": "PlanTL-GOB-ES/roberta-base-bne",
    "DistilBETO": "dccuchile/distilbert-base-spanish-uncased",
}

ID2LABEL = {
    0: "negativo",
    1: "neutro",
    2: "positivo",
}

LABEL2ID = {
    "negativo": 0,
    "neutro": 1,
    "positivo": 2,
}

In [ ]:
# ============================================================
# 3. DATASET LOCAL DE RESPALDO
# ============================================================
# Este dataset permite ejecutar la práctica sin depender de una
# descarga externa. La estructura imita un problema de sentimientos
# de 3 clases: negativo, neutro y positivo.

def crear_dataset_local():
    ejemplos = [
        # Positivos
        ("Me encantó el curso, aprendí muchísimo.", 2),
        ("La explicación fue clara y muy útil.", 2),
        ("Estoy muy satisfecho con el resultado.", 2),
        ("El servicio fue excelente y rápido.", 2),
        ("La película me pareció maravillosa.", 2),
        ("El producto superó mis expectativas.", 2),
        ("La atención fue amable y profesional.", 2),
        ("Me siento feliz con esta experiencia.", 2),
        ("El material está muy bien organizado.", 2),
        ("La clase fue dinámica e interesante.", 2),
        ("El sistema funciona perfectamente.", 2),
        ("La comida estuvo deliciosa.", 2),
        ("Recomendaría este servicio sin dudarlo.", 2),
        ("El equipo hizo un gran trabajo.", 2),
        ("La aplicación es intuitiva y útil.", 2),
        ("Me gustó mucho la presentación.", 2),
        ("La respuesta fue rápida y precisa.", 2),
        ("El ambiente fue agradable.", 2),
        ("La experiencia fue muy positiva.", 2),
        ("Estoy contento con la compra.", 2),
        # Neutros
        ("El curso inicia a las nueve de la mañana.", 1),
        ("La reunión será en el aula principal.", 1),
        ("El documento contiene cinco secciones.", 1),
        ("La computadora tiene ocho gigabytes de memoria.", 1),
        ("El archivo se guardó en la carpeta indicada.", 1),
        ("La clase tendrá una duración de cuatro horas.", 1),
        ("El sistema muestra una pantalla de inicio.", 1),
        ("El informe fue enviado ayer.", 1),
        ("La actividad consiste en clasificar textos.", 1),
        ("El dataset contiene ejemplos en español.", 1),
        ("El modelo se entrenará durante varias épocas.", 1),
        ("El examen tendrá preguntas de opción múltiple.", 1),
        ("La práctica utiliza la biblioteca Transformers.", 1),
        ("La tabla muestra los resultados obtenidos.", 1),
        ("El archivo incluye los datos de entrenamiento.", 1),
        ("La sesión termina a las dos de la tarde.", 1),
        ("El programa imprime las métricas finales.", 1),
        ("El usuario escribió un comentario corto.", 1),
        ("El modelo recibe una oración como entrada.", 1),
        ("La evaluación se realizó con el conjunto de prueba.", 1),
        # Negativos
        ("No me gustó la explicación, fue confusa.", 0),
        ("El servicio fue lento y deficiente.", 0),
        ("Estoy decepcionado con el resultado.", 0),
        ("La aplicación falla constantemente.", 0),
        ("La película fue aburrida y larga.", 0),
        ("El producto llegó dañado.", 0),
        ("La atención fue grosera.", 0),
        ("Me siento frustrado con esta experiencia.", 0),
        ("El material está desordenado.", 0),
        ("La clase fue tediosa.", 0),
        ("El sistema no funciona correctamente.", 0),
        ("La comida estuvo fría y sin sabor.", 0),
        ("No recomendaría este servicio.", 0),
        ("El equipo entregó un trabajo incompleto.", 0),
        ("La aplicación es difícil de usar.", 0),
        ("La presentación fue poco clara.", 0),
        ("La respuesta fue lenta e imprecisa.", 0),
        ("El ambiente fue incómodo.", 0),
        ("La experiencia fue negativa.", 0),
        ("Estoy molesto con la compra.", 0),
    ]

    random.shuffle(ejemplos)
    data = {
        "text": [x[0] for x in ejemplos],
        "label": [x[1] for x in ejemplos],
    }

    dataset = Dataset.from_dict(data)

    split_1 = dataset.train_test_split(test_size=0.30, seed=SEED)
    split_2 = split_1["test"].train_test_split(test_size=0.50, seed=SEED)

    return DatasetDict(
        {
            "train": split_1["train"],
            "validation": split_2["train"],
            "test": split_2["test"],
        }
    )

In [ ]:
# ============================================================
# 4. CARGA OPCIONAL DE DATASET TASS O SIMILAR
# ============================================================
# El código intenta cargar un dataset tipo TASS desde HuggingFace.
# Si falla por conexión, cambios de nombres o columnas, usa el
# dataset local de respaldo.

def normalizar_etiqueta(valor):
    """Convierte etiquetas frecuentes de sentimiento a 0, 1, 2."""
    if isinstance(valor, (int, np.integer)):
        valor = int(valor)
        if valor in [0, 1, 2]:
            return valor
        # Algunos datasets usan etiquetas 1, 2, 3.
        if valor in [1, 2, 3]:
            return valor - 1
        return None

    texto = str(valor).strip().lower()

    positivos = {"p", "pos", "positive", "positivo", "positiva", "1_positive", "label_2"}
    negativos = {"n", "neg", "negative", "negativo", "negativa", "0_negative", "label_0"}
    neutros = {"neu", "neutral", "neutro", "neutra", "none", "sin sentimiento", "no sentiment", "label_1"}

    if texto in positivos:
        return 2
    if texto in neutros:
        return 1
    if texto in negativos:
        return 0

    return None


def detectar_columna(dataset, candidatos):
    columnas = dataset.column_names
    for candidato in candidatos:
        if candidato in columnas:
            return candidato
    return None


def preparar_dataset_hf(dataset_dict):
    """Convierte un dataset de HuggingFace a columnas text y label."""
    if "train" not in list(dataset_dict.keys()):
        raise ValueError("El dataset no contiene split train.")

    train_split = dataset_dict["train"]

    text_col = detectar_columna(train_split, ["text", "tweet", "content", "sentence", "review", "texto"])
    label_col = detectar_columna(train_split, ["label", "sentiment", "polarity", "class", "etiqueta"])

    if text_col is None or label_col is None:
        raise ValueError(f"No se detectaron columnas válidas. Columnas disponibles: {train_split.column_names}")

    def mapear(batch):
        textos = batch[text_col]
        etiquetas = [normalizar_etiqueta(x) for x in batch[label_col]]
        return {"text": textos, "label": etiquetas}

    dataset_limpio = dataset_dict.map(
        mapear,
        batched=True,
        remove_columns=train_split.column_names,
    )

    dataset_limpio = dataset_limpio.filter(lambda x: x["label"] is not None)

    if len(dataset_limpio["train"]) == 0:
        raise ValueError(
            "El dataset externo quedó vacío después de normalizar etiquetas. "
            "Se usará dataset local de respaldo."
        )

    if "validation" not in dataset_limpio:
        temp = dataset_limpio["train"].train_test_split(test_size=0.30, seed=SEED)
        temp2 = temp["test"].train_test_split(test_size=0.50, seed=SEED)
        dataset_limpio = DatasetDict(
            {
                "train": temp["train"],
                "validation": temp2["train"],
                "test": temp2["test"],
            }
        )
    elif "test" not in dataset_limpio:
        temp = dataset_limpio["validation"].train_test_split(test_size=0.50, seed=SEED)
        dataset_limpio = DatasetDict(
            {
                "train": dataset_limpio["train"],
                "validation": temp["train"],
                "test": temp["test"],
            }
        )

    return dataset_limpio


def cargar_dataset():
    """Carga dataset local por defecto o dataset externo opcional.

    La versión anterior podía cargar un dataset externo, filtrar todas las
    etiquetas y dejar train/validation/test con 0 filas. Esta versión evita
    ese problema: si el dataset externo falla o queda vacío, usa el dataset
    local de respaldo.
    """

    if not USAR_DATASET_EXTERNO:
        print("\nUsando dataset local de respaldo.")
        return crear_dataset_local()

    print("\nIntentando cargar dataset tipo TASS desde HuggingFace...")

    candidatos = [
        ("iberbench/iberbench_all", "tass-tass-sentiment_analysis-2020-spanish-spain"),
        ("iberbench/iberbench_all", "iberlef-tass-sentiment_analysis-2020-spanish-spain"),
    ]

    for dataset_name, config_name in candidatos:
        try:
            print(f"Probando: {dataset_name} | config: {config_name}")
            raw_dataset = load_dataset(dataset_name, config_name)
            dataset_limpio = preparar_dataset_hf(raw_dataset)

            if len(dataset_limpio["train"]) == 0:
                raise ValueError("El dataset externo quedó vacío.")

            print("Dataset externo cargado correctamente.")
            return dataset_limpio

        except Exception as e:
            print(f"No se pudo usar {dataset_name} / {config_name}")
            print("Motivo:", str(e)[:300])

    print("\nNo se pudo usar dataset externo. Usando dataset local de respaldo.")
    return crear_dataset_local()


dataset = cargar_dataset()

print("\nDataset cargado:")
print(dataset)

if len(dataset["train"]) == 0:
    raise RuntimeError(
        "El dataset de entrenamiento está vacío. "
        "Cambia USAR_DATASET_EXTERNO = False o revisa las etiquetas del dataset externo."
    )

print("\nEjemplo de entrenamiento:")
print(dataset["train"][0])
print("\nDistribución de etiquetas en train:")
print(pd.Series(dataset["train"]["label"]).value_counts().sort_index())

In [ ]:
# ============================================================
# 5. LIMITAR TAMAÑO DEL DATASET PARA MODO RÁPIDO
# ============================================================

def limitar_dataset(dataset_dict):
    nuevo = {}
    for split in dataset_dict.keys():
        max_samples = MAX_TRAIN_SAMPLES if split == "train" else MAX_EVAL_SAMPLES
        n = min(len(dataset_dict[split]), max_samples)
        if n == 0:
            raise RuntimeError(f"El split {split} está vacío.")
        nuevo[split] = dataset_dict[split].shuffle(seed=SEED).select(range(n))
    return DatasetDict(nuevo)


dataset = limitar_dataset(dataset)
print("\nDataset después de limitar tamaño:")
print(dataset)

In [ ]:
# ============================================================
# 6. MÉTRICAS
# ============================================================

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average="macro", zero_division=0)
    return {"accuracy": acc, "f1_macro": f1_macro}

In [ ]:
# ============================================================
# 7. ARGUMENTOS DE ENTRENAMIENTO COMPATIBLES
# ============================================================
# Algunas versiones de transformers usan eval_strategy y otras
# evaluation_strategy. Esta función hace el código más robusto.

def crear_training_arguments(output_dir, model_name):
    params = inspect.signature(TrainingArguments.__init__).parameters

    args = {
        "output_dir": str(output_dir),
        "learning_rate": 2e-5,
        "per_device_train_batch_size": TRAIN_BATCH_SIZE,
        "per_device_eval_batch_size": EVAL_BATCH_SIZE,
        "num_train_epochs": NUM_EPOCHS,
        "weight_decay": 0.01,
        "logging_steps": 5,
        "save_strategy": "no",
        "report_to": "none",
        "seed": SEED,
        "fp16": torch.cuda.is_available(),
    }

    if "eval_strategy" in params:
        args["eval_strategy"] = "epoch"
    else:
        args["evaluation_strategy"] = "epoch"

    if "logging_strategy" in params:
        args["logging_strategy"] = "steps"

    return TrainingArguments(**args)

In [ ]:
# ============================================================
# 8. FUNCIÓN PRINCIPAL DE FINE-TUNING
# ============================================================

def tokenizar_dataset(dataset_dict, tokenizer):
    def tokenizar(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=MAX_LENGTH,
        )

    tokenized = dataset_dict.map(tokenizar, batched=True)
    tokenized = tokenized.rename_column("label", "labels")

    cols_a_quitar = [
        col
        for col in tokenized["train"].column_names
        if col not in ["input_ids", "attention_mask", "token_type_ids", "labels"]
    ]

    return tokenized.remove_columns(cols_a_quitar)


def entrenar_y_evaluar(nombre_modelo, checkpoint, dataset_dict):
    print("\n" + "=" * 70)
    print(f"Entrenando modelo: {nombre_modelo}")
    print(f"Checkpoint: {checkpoint}")
    print("=" * 70)

    output_dir = OUTPUT_BASE_DIR / nombre_modelo
    output_dir.mkdir(exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(checkpoint, use_fast=True)
    tokenized_dataset = tokenizar_dataset(dataset_dict, tokenizer)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint,
        num_labels=3,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    )

    training_args = crear_training_arguments(output_dir, nombre_modelo)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    inicio = time.perf_counter()
    train_output = trainer.train()
    fin = time.perf_counter()
    tiempo_entrenamiento = fin - inicio

    print(f"\nTiempo de entrenamiento de {nombre_modelo}: {tiempo_entrenamiento:.2f} segundos")

    print("\nEvaluación en conjunto de validación:")
    val_metrics = trainer.evaluate(tokenized_dataset["validation"])
    print(val_metrics)

    print("\nEvaluación en conjunto de prueba:")
    test_output = trainer.predict(tokenized_dataset["test"])

    test_logits = test_output.predictions
    test_labels = test_output.label_ids
    test_preds = np.argmax(test_logits, axis=-1)

    test_accuracy = accuracy_score(test_labels, test_preds)
    test_f1_macro = f1_score(test_labels, test_preds, average="macro", zero_division=0)

    print("\nReporte de clasificación:")
    print(
        classification_report(
            test_labels,
            test_preds,
            target_names=[ID2LABEL[i] for i in range(3)],
            zero_division=0,
        )
    )

    print("Matriz de confusión:")
    print(confusion_matrix(test_labels, test_preds))

    resultado = {
        "modelo": nombre_modelo,
        "checkpoint": checkpoint,
        "test_accuracy": test_accuracy,
        "test_f1_macro": test_f1_macro,
        "train_runtime_seg": tiempo_entrenamiento,
        "train_loss_final": train_output.training_loss,
    }

    modelo_final_dir = output_dir / "modelo_final"
    trainer.save_model(str(modelo_final_dir))
    tokenizer.save_pretrained(str(modelo_final_dir))
    print(f"\nModelo guardado en: {modelo_final_dir}")

    return {
        "resultado": resultado,
        "log_history": trainer.state.log_history,
        "modelo_dir": str(modelo_final_dir),
        "tokenizer": tokenizer,
        "trainer": trainer,
    }


In [ ]:
# ============================================================
# 9. ENTRENAR TODOS LOS MODELOS
# ============================================================

resultados = []
logs_por_modelo = {}
modelos_guardados = {}

for nombre, checkpoint in MODELS.items():
    try:
        paquete = entrenar_y_evaluar(nombre, checkpoint, dataset)
        resultados.append(paquete["resultado"])
        logs_por_modelo[nombre] = paquete["log_history"]
        modelos_guardados[nombre] = paquete["modelo_dir"]
        torch.cuda.empty_cache()
    except Exception as e:
        print("\n" + "!" * 70)
        print(f"No se pudo entrenar {nombre}")
        print("Error:", e)
        print("!" * 70)

In [ ]:
# ============================================================
# 10. TABLA COMPARATIVA DE RESULTADOS
# ============================================================

df_resultados = pd.DataFrame(resultados)

print("\n" + "=" * 70)
print("RESULTADOS COMPARATIVOS")
print("=" * 70)
print(df_resultados)

df_resultados.to_csv(
    OUTPUT_BASE_DIR / "comparativa_modelos.csv",
    index=False,
    encoding="utf-8",
)

print(f"\nTabla guardada en: {OUTPUT_BASE_DIR / 'comparativa_modelos.csv'}")

In [ ]:
# ============================================================
# 11. GRÁFICA COMPARATIVA DE MÉTRICAS
# ============================================================

if len(df_resultados) > 0:
    plt.figure(figsize=(9, 5))

    x = np.arange(len(df_resultados))
    width = 0.35

    plt.bar(x - width / 2, df_resultados["test_accuracy"], width, label="Accuracy")
    plt.bar(x + width / 2, df_resultados["test_f1_macro"], width, label="F1 macro")

    plt.xticks(x, df_resultados["modelo"], rotation=20)
    plt.ylim(0, 1.05)
    plt.ylabel("Puntuación")
    plt.title("Comparación de accuracy y F1 macro")
    plt.legend()
    plt.tight_layout()

    plt.savefig(OUTPUT_BASE_DIR / "comparacion_metricas.png", dpi=150)
    plt.show()

In [ ]:
# ============================================================
# 12. GRÁFICA COMPARATIVA DE TIEMPO
# ============================================================

if len(df_resultados) > 0:
    plt.figure(figsize=(8, 5))
    plt.bar(df_resultados["modelo"], df_resultados["train_runtime_seg"])
    plt.ylabel("Tiempo de entrenamiento, segundos")
    plt.title("Comparación de tiempo de entrenamiento")
    plt.xticks(rotation=20)
    plt.tight_layout()

    plt.savefig(OUTPUT_BASE_DIR / "comparacion_tiempo.png", dpi=150)
    plt.show()

In [ ]:
# ============================================================
# 13. CURVAS DE APRENDIZAJE
# ============================================================

def extraer_curvas(log_history):
    train_steps = []
    train_losses = []
    eval_steps = []
    eval_losses = []

    for item in log_history:
        if "loss" in item and "epoch" in item:
            train_steps.append(item["epoch"])
            train_losses.append(item["loss"])

        if "eval_loss" in item and "epoch" in item:
            eval_steps.append(item["epoch"])
            eval_losses.append(item["eval_loss"])

    return train_steps, train_losses, eval_steps, eval_losses


for nombre, log_history in logs_por_modelo.items():
    train_steps, train_losses, eval_steps, eval_losses = extraer_curvas(log_history)

    plt.figure(figsize=(8, 5))

    if len(train_losses) > 0:
        plt.plot(train_steps, train_losses, marker="o", label="Train loss")

    if len(eval_losses) > 0:
        plt.plot(eval_steps, eval_losses, marker="o", label="Validation loss")

    plt.xlabel("Época")
    plt.ylabel("Loss")
    plt.title(f"Curva de aprendizaje - {nombre}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    nombre_archivo = f"curva_aprendizaje_{nombre}.png"
    plt.savefig(OUTPUT_BASE_DIR / nombre_archivo, dpi=150)
    plt.show()

In [ ]:
# ============================================================
# 14. PREDICCIÓN CON TEXTOS NUEVOS
# ============================================================

def probar_modelo_guardado(modelo_dir, textos):
    device_id = 0 if torch.cuda.is_available() else -1

    clf = pipeline(
        "text-classification",
        model=modelo_dir,
        tokenizer=modelo_dir,
        device=device_id,
    )

    return clf(textos, truncation=True)


textos_prueba = [
    "Me encantó la clase, entendí muy bien el tema.",
    "El ejercicio fue difícil, pero útil.",
    "No me gustó el material, estaba muy confuso.",
    "La sesión comienza mañana a las diez.",
    "El modelo funcionó mucho mejor de lo esperado.",
    "Estoy decepcionado con los resultados obtenidos.",
]

print("\n" + "=" * 70)
print("PREDICCIONES CON LOS MODELOS FINE-TUNEADOS")
print("=" * 70)

for nombre, modelo_dir in modelos_guardados.items():
    print(f"\nModelo: {nombre}")
    predicciones = probar_modelo_guardado(modelo_dir, textos_prueba)

    for texto, pred in zip(textos_prueba, predicciones):
        print(f"Texto: {texto}")
        print(f"Predicción: {pred}")
        print("-" * 60)